In [3]:
#Importing dependencies
from flask import Flask, request, jsonify
#Joblib to load the mode
import joblib
import os

In [4]:
#Initialize flask app
app = Flask(__name__)

In [6]:
#Audio ML model for pkl 
model = joblib.load("/users/imbahndu/Desktop/Columbia DBM/New sound recording/audio.pkl")

In [7]:
#App route
@app.route("/")

#Testing home page 
def home():
    return "Audio assessment is running"

In [12]:
#Defining functions for feature extraction so that it can be put in the audio

#Importing the necessary modules for sound extractions
import numpy as np
import parselmouth 
import librosa
from scipy.stats import kurtosis
#from utils.tqwt import tqwt_decompose

#Feature names to match the model feature inputs 
FEATURE_ORDER = [
    "tqwt_TKEO_mean_dec_36",
    "std_6th_delta",
    "tqwt_TKEO_std_dec_13",
    "tqwt_minValue_dec_26",
    "tqwt_TKEO_mean_dec_6",
    "tqwt_entropy_shannon_dec_9",
    "ppq5Jitter",
    "tqwt_TKEO_mean_dec_26",
    "qwt_kurtosisValue_dec_16",
    "std_delta_delta_log_energy",
    "tqwt_minValue_dec_21",
    "tqwt_kurtosisValue_dec_27", 
    "mean_MFCC_2nd_coef",
    "tqwt_entropy_log_dec_1",
    "tqwt_entropy_shannon_dec_19", 
    "tqwt_energy_dec_6",
    "std_9th_delta_delta",
    "Ed2_5_coef",
    "qwt_kurtosisValue_dec_14", 
    "tqwt_energy_dec_26"
]

'''
Functions to extract the top 20 selected features from a .wav file for PD Detection, and return key pairs
'''
#loading the audio
def load_audio(file_path):
    #Using the librosa package
    y, sr = librosa.load(file_path, sr = None)

In [16]:
#extract jitter and shimmer
def extract_jitter_shimmer(file_path):
    #Caling the method for sound extraction
    snd = parselmouth.Sound(file_path)
    point_process = parselmouth.praat.call(snd, "To PointProcess", 75, 500)

    #jitter
    jitter_ppq5 = parselmouth.pratt.call([snd, point_process], "Get jitter (ppq5)", 0, 0, 75, 500, 1.3 )

    #shimmer
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 75, 500, 1.3, 1.6)

    return jitter_ppq5, shimmer_local

In [17]:
#-----
# MFCC ish features 
#----

def extract_mfcc(y, sr):
    #Extract the 13 MFCC features and computers mean values
    #Args:( y = audio signal, and sr is sample rate from audio recorder fil)
    mffc = librosa.feature.mfcc(y = y, sr=sr, n_mfcc = 13)
    mfcc_means = np.mean(mfcc, axis = 1)
    return mffc_means 

In [18]:
def tqwt_features( y , levels = 36):
    #Mimisc TQWT featurs by performing wavelet decomposition 
    '''
    Shannon entropy, energy, kurtosis, min value

    Args (audio signal. number of decomposition levels)
    '''

    coeffs = pywt.wavedec(y, "db4", level = levels)
    features = {}

    #enumerating the coeffs
    for i, c in enumerate(coeffs):
        features["tqwt_energy_dec_" + i] = np.sum(c**2)
        features["tqwt_entropy_shannon_dec_" + i] = entropy(np.abs(c)+1**-10)
        features["tqwt_kurtosisValue_dec_"+ i] = kurtosis(c)
        features["tqwt_minValue_dec_" + i] = np.min(c)

        # Ensure all levels up to 36 are present (placeholder fill)
    for lvl in range(levels + 1):  # levels = 36
        for key in ["energy", "entropy_shannon", "entropy_log", "kurtosisValue", "minValue", "TKEO_mean", "TKEO_std"]:
            feature_name = f"tqwt_{key}_dec_{lvl}"
            if feature_name not in features:
                features[feature_name] = 0

    return features

In [19]:
'''
    FULL FEATURE EXTRACTION PIPELINE
    '''
def extract_features(file_path):
    y, r = load_audio(file_path, sr = None)
    return y, sr

    #Calling all the previously defined functions

    jitter_ppq5 , shimmer_local = extract_jitter_shimmer(file_path)
    mfcc_means = extract_mfcc(y, sr)
    tqwt_feats = tqwt_features(y)

    #Combine all featurs into a single dictionary
    feature_vector = {
        "ppq5jitter": jitter_ppq5,
        "shimmer_local": shimmer_local,
        "mean_MFCC_2nd_coef": mgcc_means[1], 
    }

    '''
    SELECT ONLY TRAINED FEATURES
    '''
    def extract_selected_features(file_path):
        #Extract top 20 featues that were used in training
        selected_features = [
        "tqwt_TKEO_mean_dec_36", "std_6th_delta", "tqwt_energy_dec_33",
        "tqwt_entropy_log_dec_33", "tqwt_energy_dec_32", "tqwt_entropy_shannon_dec_33",
        "tqwt_energy_dec_34", "tqwt_entropy_shannon_dec_32", "tqwt_energy_dec_31",
        "tqwt_entropy_shannon_dec_34", "tqwt_energy_dec_35", "tqwt_entropy_log_dec_32",
        "tqwt_energy_dec_30", "tqwt_entropy_shannon_dec_31", "tqwt_entropy_log_dec_31",
        "tqwt_entropy_log_dec_34", "tqwt_energy_dec_29", "tqwt_entropy_shannon_dec_30",
        "tqwt_entropy_log_dec_30", "tqwt_entropy_log_dec_29"
    ]
    
    full_features = extract_features(file_path)
    
    # Safely select features (if missing, fill with 0)
    selected = {f: full_features.get(f, 0) for f in selected_features}
    
    return pd.DataFrame([selected])


In [20]:
#Route 1: Accept raw audio (.wav) uploaded from app

@app.route("/predict-audio", methods=["POST"])

#predict method from audio
def predict_from_audio():

    #checking if the uploaded file is in the request directory in audio_page.dart
    if "file" not in request.files:
        return jsonify({"Error: No audio file provided"}), 400
    
    #Getting uploaded file
    file = request.files["file"]

    #Save temporarily to uploads folder
    filepath = os.path.join('uploads', file.filename)
    file.save(filepath)

    #trying to extract the features 
    try:
        #Extract features from the.wav file, which has been preprocessed with librosa/parselmouth 
        features = extract_features_from_audio(filepath)

        #Optionally do preprocess here if predictions are weird

        #Make predictions on trained model
        predictions = model.predict([features])[0]

        #Return JSON response with pred
        return jsonify({'prediction': int(prediction), "probabilities": {
            "healthy": probability[0], 
            "parkinson" : probability[1]
        }})
    
    #except condition 
    except Exception as e:
        return jsonify({"Error": str(e)}), 500

    finally:
        #Delete file temporarily
        os.remove(filepath)

